<a href="https://colab.research.google.com/github/neuralnex/ASL-to-text/blob/main/asl_extended/notebooks/train_cnn_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train CNN-Based Fingerspelling Classifier


In [1]:
import sys
import os
sys.path.append('..')

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from data.cnn_data_loader import CNNDataLoader
from models.cnn_classifier import create_cnn_classifier, save_model


ModuleNotFoundError: No module named 'data'

In [ ]:
loader = CNNDataLoader(data_dir='../../dataSet')
X, y_categorical, y = loader.load_data()

print(f"Total samples: {len(X)}")
print(f"Image shape: {X.shape[1:]}")
print(f"Number of classes: {len(loader.classes)}")


In [ ]:
(X_train, y_train), (X_val, y_val), (X_test, y_test) = loader.split_data(X, y_categorical)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples: {len(X_test)}")


In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

train_generator = train_datagen.flow(X_train, y_train, batch_size=10)
val_generator = ImageDataGenerator(rescale=1./255).flow(X_val, y_val, batch_size=10)


In [ ]:
model = create_cnn_classifier(input_shape=X.shape[1:], num_classes=len(loader.classes))
model.summary()


In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=len(X_train) // 10,
    epochs=5,
    validation_data=val_generator,
    validation_steps=len(X_val) // 10,
    verbose=1
)


In [ ]:
test_loss, test_accuracy = model.evaluate(X_test / 255.0, y_test, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")


In [ ]:
save_model(
    model,
    '../models/cnn_model.json',
    '../models/cnn_model.h5'
)
print("Model saved successfully")


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Model Accuracy')

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Model Loss')

plt.tight_layout()
plt.savefig('../models/cnn_training_history.png')
plt.show()
